# ISeeSnow: Idealized topography — Coulomb only — AVAC

This notebook runs one official ISeeSnow case without calibration, writes the required peak-flow-thickness and peak-flow-velocity rasters, and compares them on the supplied grid with participating-model submissions.


## Reproducible environment

The first code cell installs the repository's validation package and Python dependencies into the active kernel. A clean checkout also needs GNU Make and gfortran to compile the selected solver on first use. The pinned official ISeeSnow 1.0 dataset is downloaded automatically on first use.


In [ ]:
from pathlib import Path
SEARCH_ROOT = Path.cwd().resolve()
REPOSITORY = next(candidate for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents) if (candidate / 'validation' / 'pyproject.toml').is_file())
%pip install -q -e {REPOSITORY / 'validation'}


In [ ]:
import os
from avac4qgis_validation import validation_case
case = validation_case('ISeeSnow', 'CoulombOnly')
CORES = max(1, os.cpu_count() or 1)
case.path


In [ ]:
from avac4qgis_validation.datasets import ensure_iseesnow
benchmark = ensure_iseesnow()
benchmark


## Prescribed benchmark configuration

The idealized case is repeated with $\mu=0.4$ and the turbulent resistance disabled, following the ISeeSnow Coulomb-only protocol. No peer-model result is used to select an AVAC parameter. The simulation ceiling is 1200 s and the native state is checked for practical arrest.


In [ ]:
CASE_NAME = 'CoulombOnly'
case.run(case.path.parent / 'run_iseesnow_avac.py', '--case', CASE_NAME, '--workers', CORES, '--spatial-order', 2, '--overwrite', cwd=case.path.parent)


## Run diagnostics and ISeeSnow submission

The summary records volume, duration, practical stop time, numerical controls, solver hash, and generated standard-format files.


In [ ]:
summary = case.json('run_summary.json')
summary


## Direct peer comparison

Only peer fields with exactly matching dimensions, cell size, and cell-center coordinates are included; no shifting, clipping, padding, or resampling is performed.


In [ ]:
case.run(case.path.parent / 'compare_iseesnow.py', '--case', CASE_NAME, cwd=case.path.parent)
case.show(f'../plots/{CASE_NAME}_pft_peer_comparison.png', f'../plots/{CASE_NAME}_pfv_peer_comparison.png', f'../plots/{CASE_NAME}_scalar_peer_comparison.png')
